# Deep Learning Project — Milestone 1

## Reproducing SIB-200 with XLM-R Base

In this project, we reproduce the SIB-200 topic classification results using the XLM-R base model.

We use the English dataset, which contains 701 training examples, 99 validation examples, and 204 test examples.

Our goal is to train the model using five different random seeds, calculate the average test accuracy, and compare our results with the authors' reported accuracy of 90.0%.

We also fix compatibility issues in the original training code and use mixed precision to improve training efficiency.

This notebook includes the full setup, training, evaluation, and results so the experiment can be reproduced.

In [66]:

# Check GPU and mixed precision support

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Please enable a GPU."

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True


## 2. Environment Setup

We install the libraries needed to run the original SIB-200 code. We also use a GPU with PyTorch mixed precision to make training faster.

In [67]:

# Install the libraries needed for this project

%pip install -q transformers==5.1.0 datasets scikit-learn sentencepiece pandas


In [68]:

# Download the original SIB-200 repository

from pathlib import Path
import subprocess

repo = Path("/content/sib-200")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/dadelani/sib-200.git",
            str(repo)
        ],
        check=True
    )

print("Repository downloaded:", repo.exists())
print("Training script:", (repo / "code/train_textclass.py").exists())
print("English dataset:",
      (repo / "data/annotated/eng_Latn/train.tsv").exists())


Repository downloaded: True
Training script: True
English dataset: True


In [69]:


# Record dependency versions and repository commit

import transformers
import sklearn
import subprocess

print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)

commit = subprocess.check_output(
    ["git", "-C", "/content/sib-200", "rev-parse", "HEAD"],
    text=True
).strip()

print("SIB-200 commit:", commit)


Transformers: 5.1.0
Scikit-learn: 1.6.1
SIB-200 commit: efad32e0fc601cfbfdf5ca80672f29697f6f401f



## 3. Compatibility Fixes

The original SIB-200 code uses some older PyTorch and Transformers functions.

We apply the four fixes required by the professor so the training code works in our Colab environment.


In [70]:

# Restore the original code before applying our fixes

from pathlib import Path
import subprocess

repo = Path("/content/sib-200")

subprocess.run(
    [
        "git", "-C", str(repo), "restore",
        "code/train_textclass.py",
        "code/util_textclass.py"
    ],
    check=True
)

print("Original training files restored!")


Original training files restored!


### 3.1 Fix AdamW Import

The original code imports AdamW from Transformers, but this is no longer supported in newer versions.

We change the import to use PyTorch's AdamW optimizer instead.

In [71]:

# Fix 1 - Import AdamW from PyTorch

from pathlib import Path
import re

train_file = Path("/content/sib-200/code/train_textclass.py")
code = train_file.read_text()

# Remove AdamW from the Transformers import
code, count = re.subn(
    r"(?m)^[ \t]*AdamW,[ \t]*\n",
    "",
    code
)

assert count == 1, "Could not find the old AdamW import."

# Add the PyTorch import
if "from torch.optim import AdamW" not in code:
    code = "from torch.optim import AdamW\n" + code

train_file.write_text(code)

print("AdamW import fixed!")
print("PyTorch AdamW:", "from torch.optim import AdamW" in code)


AdamW import fixed!
PyTorch AdamW: True


### 3.2 Fix Tokenizer Encoding

The original code uses `encode_plus` to tokenize the text.

We replace it with a direct tokenizer call, which is the recommended approach in newer versions of Transformers.

In [72]:

# Fix 2 - Replace encode_plus with a direct tokenizer call

from pathlib import Path

util_file = Path("/content/sib-200/code/util_textclass.py")
code = util_file.read_text()

old = "tokenizer.encode_plus(text=instance.text,"
new = "tokenizer(text=instance.text,"

assert code.count(old) == 1, "Could not find the old tokenizer call."

code = code.replace(old, new, 1)
util_file.write_text(code)

print("Tokenizer encoding fixed!")
print("Direct tokenizer call:", new in code)


Tokenizer encoding fixed!
Direct tokenizer call: True


### 3.3 Fix Tokenizer Padding

The original code uses `pad_to_max_length=True`, which is deprecated in newer versions of Transformers.

We replace it with `padding="max_length"` to keep all tokenized inputs at the same length.

In [73]:

# Fix 3 - Update the tokenizer padding argument

from pathlib import Path

util_file = Path("/content/sib-200/code/util_textclass.py")
code = util_file.read_text()

old = "pad_to_max_length=True"
new = 'padding="max_length"'

assert code.count(old) == 1, "Could not find the old padding argument."

code = code.replace(old, new, 1)
util_file.write_text(code)

print("Tokenizer padding fixed!")
print("New padding argument:", new in code)


Tokenizer padding fixed!
New padding argument: True


### 3.4 Fix Cached Feature Loading

Newer versions of PyTorch use safer settings when loading saved files.

The original SIB-200 code needs to load cached dataset features, so we add `weights_only=False` for these locally generated files.

In [74]:

# Fix 4 - Allow loading cached dataset features

from pathlib import Path

train_file = Path("/content/sib-200/code/train_textclass.py")
code = train_file.read_text()

old = "torch.load(cached_features_file)"
new = "torch.load(cached_features_file, weights_only=False)"

assert code.count(old) == 1, "Could not find the old cache loading code."

code = code.replace(old, new, 1)
train_file.write_text(code)

print("Cached feature loading fixed!")
print("New loading option:", new in code)


Cached feature loading fixed!
New loading option: True


## 4. Training Optimization

We use BF16 mixed precision on the NVIDIA A100 GPU to make training faster.

The A100 supports BF16, which allows us to reduce computation time while keeping the model architecture and training hyperparameters unchanged.

### 4.1 Prepare BF16 Training

We keep the corrected FP32 training script and create a separate version for BF16.

We use PyTorch automatic mixed precision with `autocast` to improve training efficiency while keeping the same model and hyperparameters.

In [75]:
# Create a separate script for BF16 training

from pathlib import Path
import shutil

code_dir = Path("/content/sib-200/code")

original = code_dir / "train_textclass.py"
amp_script = code_dir / "train_textclass_amp.py"

assert original.exists(), "Training script not found."

# Copy the corrected FP32 script
shutil.copy2(original, amp_script)

print("FP32 script:", original.exists())
print("BF16 copy:", amp_script.exists())


FP32 script: True
BF16 copy: True


In [76]:

# Add BF16 mixed precision to the training script

from pathlib import Path
import ast
import re
import torch

assert torch.cuda.is_available()
assert torch.cuda.is_bf16_supported()

code_dir = Path("/content/sib-200/code")
original = code_dir / "train_textclass.py"
amp_file = code_dir / "train_textclass_amp.py"

# Start from the corrected FP32 script
code = original.read_text()

def patch(pattern, replacement):
    global code
    code, count = re.subn(
        pattern, replacement, code, count=1, flags=re.MULTILINE
    )
    assert count == 1, f"Patch failed: {pattern}"

# 1. Use BF16 for the forward pass and loss calculation
patch(
    r"^([ \t]*)outputs = model\(\*\*inputs\)\n"
    r"[ \t]*loss = outputs\[0\]",
    lambda m: (
        f"{m.group(1)}with torch.autocast('cuda', dtype=torch.bfloat16):\n"
        f"{m.group(1)}    outputs = model(**inputs)\n"
        f"{m.group(1)}    loss = outputs[0]"
    )
)

# BF16 normally does not need GradScaler.
# Backward, optimizer step and gradient clipping remain unchanged.

# Check the modified script before saving
ast.parse(code)
amp_file.write_text(code)

print("BF16 script updated successfully!")
print("Syntax check: OK")
print("BF16 autocast:", "torch.bfloat16" in code)
print("FP32 script preserved:", original.exists())


BF16 script updated successfully!
Syntax check: OK
BF16 autocast: True
FP32 script preserved: True


## 5. Dataset Preparation

We use the English dataset (`eng_Latn`) from SIB-200 for topic classification.

The dataset contains 701 training examples, 99 validation examples, and 204 test examples.

We check the dataset size and look at a few training examples before starting the experiments.

In [77]:

# Load and check the English SIB-200 dataset

from pathlib import Path
import pandas as pd

data_dir = Path("/content/sib-200/data/annotated/eng_Latn")

train_df = pd.read_csv(data_dir / "train.tsv", sep="\t")
val_df = pd.read_csv(data_dir / "dev.tsv", sep="\t")
test_df = pd.read_csv(data_dir / "test.tsv", sep="\t")

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Test examples:", len(test_df))

print("\nColumns:", train_df.columns.tolist())

print("\nTraining examples:")
display(train_df.head(3))


Training examples: 701
Validation examples: 99
Test examples: 204

Columns: ['index_id', 'category', 'text']

Training examples:


,index_id,category,text
0,431,geography,Turkey is encircled by seas on three sides: th...
1,403,science/technology,In the beginning of the war they mostly travel...
2,1592,science/technology,"As knowledge of Greek declined, the West found..."


## 6. Model Configuration

We use XLM-R base to reproduce the English SIB-200 topic classification results.

We first examine the authors' original training scripts to identify the reported hyperparameters. Since the available scripts do not fully specify the configuration used for the archived English XLM-R base results, we clearly document our selected settings.

We use BF16 mixed precision on the NVIDIA A100 GPU to improve training efficiency. All five experiments use the same configuration, except for the random seed.

### 6.1 Inspect Original Training Settings

First, we check the training scripts in the authors' repository to find the settings they used.

Most scripts, including `glot.sh` and `xlmr_zero.sh`, use 20 epochs, batch size 16, gradient accumulation 2, maximum sequence length 164, and learning rate 1e-5.

However, `xlmr_all.sh` uses 10 epochs and batch size 32. Since the scripts use different settings, we compare them before choosing our configuration.

The exact settings for the authors' XLM-R base results are not fully documented.

In [78]:

# Check the settings in the original training scripts

from pathlib import Path

code_dir = Path("/content/sib-200/code")

keywords = [
    "MAX_LENGTH=", "BATCH_SIZE=", "NUM_EPOCHS=",
    "BERT_MODEL=", "--max_seq_length",
    "--per_gpu_train_batch_size", "--num_train_epochs",
    "--learning_rate", "--gradient_accumulation_steps"
]

for script in sorted(code_dir.glob("*.sh")):
    print(f"\n--- {script.name} ---")

    for line in script.read_text().splitlines():
        if any(key in line for key in keywords):
            print(line.strip())




--- afriberta.sh ---
export MAX_LENGTH=164
export BATCH_SIZE=16
export NUM_EPOCHS=20
export BERT_MODEL=castorini/afriberta_large
--max_seq_length  $MAX_LENGTH \
--num_train_epochs $NUM_EPOCHS \
--learning_rate 1e-5 \
--per_gpu_train_batch_size $BATCH_SIZE \
--gradient_accumulation_steps 2 \

--- afroxlmr.sh ---
export MAX_LENGTH=164
export BATCH_SIZE=16
export NUM_EPOCHS=20
export BERT_MODEL=Davlan/afro-xlmr-large
--max_seq_length  $MAX_LENGTH \
--num_train_epochs $NUM_EPOCHS \
--learning_rate 1e-5 \
--per_gpu_train_batch_size $BATCH_SIZE \
--gradient_accumulation_steps 2 \

--- afroxlmr61L.sh ---
export MAX_LENGTH=164
export BATCH_SIZE=16
export NUM_EPOCHS=20
export BERT_MODEL=Davlan/afro-xlmr-large-61L
--max_seq_length  $MAX_LENGTH \
--num_train_epochs $NUM_EPOCHS \
--learning_rate 1e-5 \
--per_gpu_train_batch_size $BATCH_SIZE \
--gradient_accumulation_steps 2 \

--- glot.sh ---
export MAX_LENGTH=164
export BATCH_SIZE=16
export NUM_EPOCHS=20
export BERT_MODEL=cis-lmu/glot500-base
--

### 6.2 Hyperparameter Selection

We use XLM-R base because it is required for this milestone.

First, we check the authors' original training scripts. Since the assignment mentions the Glot500 script as an example, we use the settings from `glot.sh` as our reference.

The script uses 20 epochs, batch size 16, gradient accumulation 2, maximum sequence length 164, and learning rate 1e-5.

This gives an effective batch size of 32 (16 × 2).

We apply these settings to XLM-R base. Although the authors' XLM-R base training command is not available in the repository, this allows us to follow a documented training configuration.

We use BF16 mixed precision on an NVIDIA A100 GPU to make training faster.

Finally, we train the model five times using seeds 1 to 5. We calculate the average test accuracy and compare it with the authors' reported English result of 90.0%.

### 6.3 Final Training Configuration

Here, we define all the settings we will use for training.

We keep the same settings for all five runs and only change the random seed. This makes the results easier to compare and reproduce.

In [79]:


# Define the final training settings

from pathlib import Path
import torch

MODEL_NAME = "FacebookAI/xlm-roberta-base"
DATA_DIR = Path("/content/sib-200/data/annotated/eng_Latn")
TRAIN_SCRIPT = Path("/content/sib-200/code/train_textclass_amp.py")
OUTPUT_DIR = Path("/content/sib-200/final_results")

MAX_SEQ_LENGTH = 164
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 8
GRAD_ACCUMULATION = 2
EPOCHS = 20
LEARNING_RATE = 1e-5
SEEDS = [1, 2, 3, 4, 5]

EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUMULATION

assert torch.cuda.is_available(), "GPU is required."
assert torch.cuda.is_bf16_supported(), "BF16 is not supported."
assert TRAIN_SCRIPT.exists(), "Training script not found."
assert (DATA_DIR / "train.tsv").exists(), "Dataset not found."

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Precision: BF16")
print("Batch size:", TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUMULATION)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("Seeds:", SEEDS)


Model: FacebookAI/xlm-roberta-base
GPU: NVIDIA A100-SXM4-40GB
Precision: BF16
Batch size: 16
Gradient accumulation: 2
Effective batch size: 32
Epochs: 20
Learning rate: 1e-05
Maximum sequence length: 164
Seeds: [1, 2, 3, 4, 5]


## 7. Training Five Models

Now we train XLM-R base on the English SIB-200 dataset five times, using seeds 1 to 5.

Each run uses the same training settings and BF16 mixed precision. We save the models and training logs separately so we can check the results of each run.

We use the validation set during training. After all five models are trained, we evaluate them on the test set and compare the average accuracy with the authors' result.

In [80]:

import subprocess
import sys
import time

LABELS_FILE = DATA_DIR / "labels.txt"

for seed in SEEDS:
    run_dir = OUTPUT_DIR / f"seed_{seed}"
    log_file = OUTPUT_DIR / f"seed_{seed}.log"

    command = [
        sys.executable, str(TRAIN_SCRIPT),
        "--data_dir", str(DATA_DIR),
        "--model_type", "xlmroberta",
        "--model_name_or_path", MODEL_NAME,
        "--output_dir", str(run_dir),
        "--labels", str(LABELS_FILE),
        "--max_seq_length", str(MAX_SEQ_LENGTH),
        "--per_gpu_train_batch_size", str(TRAIN_BATCH_SIZE),
        "--per_gpu_eval_batch_size", str(EVAL_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRAD_ACCUMULATION),
        "--num_train_epochs", str(EPOCHS),
        "--learning_rate", str(LEARNING_RATE),
        "--save_steps", "500000",
        "--seed", str(seed),
        "--do_train",
        "--do_eval",
        "--overwrite_output_dir",
    ]

    print(f"\nTraining seed {seed} of {len(SEEDS)}...", flush=True)
    start = time.time()

    with open(log_file, "w") as log:
        result = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    if result.returncode != 0:
        print(log_file.read_text()[-4000:])
        raise RuntimeError(f"Training failed for seed {seed}.")

    elapsed = time.time() - start
    print(f"Seed {seed} completed in {elapsed / 60:.1f} minutes.")
    print(f"Saved to: {run_dir}")



Training seed 1 of 5...
Seed 1 completed in 1.1 minutes.
Saved to: /content/sib-200/final_results/seed_1

Training seed 2 of 5...
Seed 2 completed in 1.1 minutes.
Saved to: /content/sib-200/final_results/seed_2

Training seed 3 of 5...
Seed 3 completed in 1.1 minutes.
Saved to: /content/sib-200/final_results/seed_3

Training seed 4 of 5...
Seed 4 completed in 1.1 minutes.
Saved to: /content/sib-200/final_results/seed_4

Training seed 5 of 5...
Seed 5 completed in 1.2 minutes.
Saved to: /content/sib-200/final_results/seed_5


## 8. Test Evaluation

Now we evaluate our five trained models on the English SIB-200 test set.

We calculate the test accuracy for each seed and then take the average.

Finally, we compare our average accuracy with the authors' reported result of 90.0% for English XLM-R base.

In [81]:

import subprocess
import sys
import pandas as pd

AUTHOR_ACCURACY = 90.0
results = []

for seed in SEEDS:
    run_dir = OUTPUT_DIR / f"seed_{seed}"

    # Make sure the trained model exists
    assert (run_dir / "config.json").exists(), f"Missing model for seed {seed}"

    command = [
        sys.executable, str(TRAIN_SCRIPT),
        "--data_dir", str(DATA_DIR),
        "--model_type", "xlmroberta",
        "--model_name_or_path", MODEL_NAME,
        "--output_dir", str(run_dir),
        "--labels", str(DATA_DIR / "labels.txt"),
        "--max_seq_length", str(MAX_SEQ_LENGTH),
        "--per_gpu_eval_batch_size", str(EVAL_BATCH_SIZE),
        "--output_result", "test_results",
        "--output_prediction_file", "test_predictions",
        "--do_predict",
    ]

    # Evaluate the saved model on the test set
    with open(OUTPUT_DIR / f"seed_{seed}_test.log", "w") as log:
        process = subprocess.run(
            command,
            stdout=log,
            stderr=subprocess.STDOUT
        )

    if process.returncode != 0:
        log_file = OUTPUT_DIR / f"seed_{seed}_test.log"
        print(log_file.read_text()[-4000:])
        raise RuntimeError(f"Test evaluation failed for seed {seed}")

    # Read the accuracy saved by the original script
    result_file = run_dir / "test_results.txt"
    metrics = {}

    for line in result_file.read_text().splitlines():
        if " = " in line:
            key, value = line.split(" = ", 1)
            metrics[key.strip()] = float(value.strip())

    results.append({
        "Seed": seed,
        "Test Accuracy (%)": metrics["acc"] * 100
    })

# Display results
results_df = pd.DataFrame(results)
average_accuracy = results_df["Test Accuracy (%)"].mean()

display(results_df.round(2))

print(f"\nAverage test accuracy: {average_accuracy:.2f}%")
print(f"Authors' reported accuracy: {AUTHOR_ACCURACY:.2f}%")
print(f"Difference: {average_accuracy - AUTHOR_ACCURACY:+.2f} percentage points")


,Seed,Test Accuracy (%)
0,1,89.71
1,2,88.24
2,3,89.71
3,4,90.69
4,5,91.18



Average test accuracy: 89.90%
Authors' reported accuracy: 90.00%
Difference: -0.10 percentage points


## 9. Results and Conclusion

In this milestone, we trained XLM-R base on the English SIB-200 dataset using five different random seeds.

We first checked the authors' training scripts and used the settings from glot.sh, which was mentioned as an example in the assignment.

After evaluating all five models, we got an average test accuracy of **89.90%**, compared with the authors' **90.00%**.

The difference is only **0.10 percentage points**, so our result is very close to the original benchmark.

The small difference may come from random variation, mixed precision, or differences in the training setup. We also could not confirm the exact settings used for the authors' archived XLM-R base models.

Overall, we completed the Milestone 1 requirements and reproduced a result close to the authors' reported accuracy.